# Winnipeg Rental Market Intelligence

## Census-Tract Analysis of Rental Prices, Supply, Vacancy, and Market Affordability

This notebook presents the reproducible analytical workflow for the Winnipeg rental-market project.

## 1. Research Question

**How do rental prices, vacancy rates, rental-housing supply, and local economic characteristics vary across Winnipeg, and what factors are associated with rental-market pressure and affordability?**

## 2. Data Sources

- CMHC Rental Market Survey: rental supply, average rents, and vacancy rates.
- Statistics Canada Census table 98-10-0058-01: median household income.

The primary geographic unit is the Winnipeg census tract. The CMHC rental segment used for the tract analysis is **Apartment & Other**.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path(
    "/Users/abbas90/winnipeg_rental_market_intelligence"
)

DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "winnipeg_ct_rental_market_analytical.csv"
)

df = pd.read_csv(DATA_PATH)

print("Winnipeg Rental Market Intelligence")
print("-" * 50)
print(f"Dataset: {DATA_PATH.name}")
print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns):,}")
print(f"Unique census tracts: {df['ct_id'].nunique():,}")
print(f"Duplicate CT IDs: {df['ct_id'].duplicated().sum():,}")


Winnipeg Rental Market Intelligence
--------------------------------------------------
Dataset: winnipeg_ct_rental_market_analytical.csv
Rows: 187
Columns: 25
Unique census tracts: 187
Duplicate CT IDs: 0


## 3. Data Quality

CMHC suppression and availability codes are treated as missing rather than zero. Variable coverage differs across the rental-market measures, so each analysis uses the observations available for the relevant variables.

In [2]:
print('Duplicate CT IDs:', df['ct_id'].duplicated().sum())
print('\nMissing observations by variable:')
print(df.isna().sum().sort_values(ascending=False).head(15))

Duplicate CT IDs: 0

Missing observations by variable:
vacancy_3br_plus      179
vacancy_bachelor      177
rent_3br_plus         173
rent_bachelor         162
vacancy_2br           139
vacancy_1br           138
vacancy_total         130
rent_2br              120
rent_1br              114
affordability_tier    110
annual_rent_total     110
rent_total            110
rent_income_pct       110
units_total            52
share_2br              52
dtype: int64


## 4. Rental Market Descriptives

Summary statistics for rental supply, total average rent, vacancy, and household income.

In [3]:
summary_cols = [
    'units_total',
    'rent_total',
    'vacancy_total',
    'median_household_income_2020'
]

df[summary_cols].describe().round(2)

,units_total,rent_total,vacancy_total,median_household_income_2020
count,135.0,77.00,57.00,185.00
mean,287.7,1179.99,1.82,87081.62
std,248.6,289.78,3.62,26072.36
min,4.0,632.00,0.00,37600.00
25%,85.0,982.00,0.40,69000.00
50%,223.0,1124.00,0.70,84000.00
75%,385.5,1373.00,1.80,103000.00
max,954.0,1927.00,26.00,159000.00


## 5. Market Affordability Indicator

The project uses an ecological market indicator:

**Annualized total average rent / median household income × 100**

This is a geographic market indicator and should not be interpreted as individual household rent burden.

In [4]:
affordability = df[['ct_id', 'rent_total', 'median_household_income_2020']].dropna().copy()
affordability['rent_income_pct'] = (
    affordability['rent_total'] * 12 /
    affordability['median_household_income_2020'] * 100
)

print('Observations:', len(affordability))
print(affordability['rent_income_pct'].describe().round(2))

Observations: 77
count    77.00
mean     19.95
std       5.01
min      12.44
25%      16.39
50%      19.32
75%      22.67
max      32.33
Name: rent_income_pct, dtype: float64


## 6. Correlation Analysis

Spearman correlation is used alongside Pearson correlation to assess monotonic relationships without relying exclusively on linear association.

In [5]:
from scipy.stats import pearsonr, spearmanr

rent_income = df[['rent_total', 'median_household_income_2020']].dropna()

pearson_r, pearson_p = pearsonr(
    rent_income['rent_total'],
    rent_income['median_household_income_2020']
)

spearman_rho, spearman_p = spearmanr(
    rent_income['rent_total'],
    rent_income['median_household_income_2020']
)

print(f'Pearson r = {pearson_r:.3f}, p = {pearson_p:.4g}')
print(f'Spearman rho = {spearman_rho:.3f}, p = {spearman_p:.4g}')

Pearson r = 0.577, p = 4.073e-08
Spearman rho = 0.498, p = 3.977e-06


## 7. Multivariable Rent Model

Total average rent is modelled using log-transformed rental supply and median household income.

The model is observational and coefficients are interpreted as associations, not causal effects.

In [6]:
import statsmodels.api as sm

model_data = df[
    ['units_total', 'rent_total', 'median_household_income_2020']
].dropna().copy()

model_data['log_units_total'] = np.log1p(model_data['units_total'])

X = model_data[['log_units_total', 'median_household_income_2020']]
X = sm.add_constant(X)
y = model_data['rent_total']

model = sm.OLS(y, X).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:             rent_total   R-squared:                       0.678
Model:                            OLS   Adj. R-squared:                  0.666
Method:                 Least Squares   F-statistic:                     58.87
Date:                Thu, 03 Sep 2026   Prob (F-statistic):           1.71e-14
Time:                        15:45:07   Log-Likelihood:                -385.47
No. Observations:                  59   AIC:                             776.9
Df Residuals:                      56   BIC:                             783.2
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                                   coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------------------
const           

## 8. Model Diagnostics

Diagnostics include residual normality, heteroskedasticity, multicollinearity, and influential observations.

In [7]:
from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.stats.outliers_influence import variance_inflation_factor

bp = het_breuschpagan(model.resid, model.model.exog)
print('Breusch-Pagan p-value:', round(bp[1], 4))

vif = pd.DataFrame({
    'variable': X.columns,
    'VIF': [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
})
print('\nVIF:')
print(vif.round(3))

Breusch-Pagan p-value: 0.2373

VIF:
                       variable    VIF
0                         const  1.000
1               log_units_total  1.009
2  median_household_income_2020  1.009


## 9. Interpretation

The final model explains approximately 68% of the variation in observed total average rent across the model sample. Higher median household income and larger rental-market inventories are positively associated with total average rent after accounting for the other predictor.

These relationships should not be interpreted as evidence of causation.

## 10. Limitations

- CMHC variables have different geographic coverage.
- Suppressed and unavailable values reduce sample sizes for some analyses.
- The affordability indicator is ecological rather than household-level.
- The analysis is cross-sectional and observational.
- The Manitoba rural/small-centre benchmark uses a much smaller sample and a different geographic scale.